# Vector Database : ChromaDB사용
- cos유사도 사용

In [1]:
!pip install chromadb sentence-transformers # sentence-transformers : 임베딩 모델

In [2]:
import chromadb
from chromadb import PersistentClient # 크로마DB 클라이언트 객체(DB 접속 관리자 역할을 함)
# !pip show chromadb # 설치 확인하기
print(chromadb.__file__) # 패키지 경로 확인하기

/usr/local/lib/python3.12/dist-packages/chromadb/__init__.py


In [22]:
client = PersistentClient(path=".chroma") # 디렉토리 지정하기
!ls -a

# Collection 생성하기 ≈ SQL Table
collection = client.get_or_create_collection("test") # collenction 읽기 / 만들기

# 데이터 벡터화 후 저장하기
print("[데이터 벡터화 후 저장]")
texts = ["Hello world", "Chroma is cool"] # list로 넣어야함
ids = ["doc1", "doc2"]
metas = [{"source":"greeting"},{"source":"statement"}] # dict형태로 list에 넣어야 함

# 현재 컬렉션에 설정된 임베딩 함수 가져오기
# chroma가 문자를 숫자벡터로 변환함 -> 기본 내장 임베딩 모델 : all-MiniLM-L6-v2지원
embedding_fn = collection._embedding_function

embeddings = embedding_fn(texts)
print(embeddings[0][:5])
print("type=",type(embeddings),"| shape = ",(len(embeddings),len(embeddings[0])))
print()

for i, vector in enumerate(embeddings):
  print(f"문서 : {texts[i]}")
  print(f"임베딩 값 5개만 : {vector[:5]}") # 384차원의 float32 벡터로 변환 (여기서의 차원은 class의 갯수이다.)
  print(f'차원수(class갯수) : {len(vector)}')
  print("-" * 30)
print()

# 참고 : 두 문장간 유사도 계산하기
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity([embeddings[0]],[embeddings[1]])[0][0]
print(f"두 문장(벡터)간 코사인 유사도 : {sim:.4f}") # 1에 가까울 수 록 관련 O, 0에 가까울 수 록 관련 X
print()

# Collection에 저장
collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=metas
)
print("Collection Counts :",collection.count())

.  ..  .chroma	.config  sample_data
[데이터 벡터화 후 저장]
[-0.03447729  0.03102317  0.00673498  0.02610895 -0.03936207]
type= <class 'list'> | shape =  (2, 384)

문서 : Hello world
임베딩 값 5개만 : [-0.03447729  0.03102317  0.00673498  0.02610895 -0.03936207]
차원수(class갯수) : 384
------------------------------
문서 : Chroma is cool
임베딩 값 5개만 : [-0.11315749 -0.00511692 -0.01389769 -0.02509309 -0.04774468]
차원수(class갯수) : 384
------------------------------

두 문장(벡터)간 코사인 유사도 : 0.0580

Collection Counts : 2


In [45]:
# 저장된 문서 조회
results = collection.get(include=["documents","metadatas"])
print(results)
print()

for doc, meta, id in zip(results['documents'], results['metadatas'], results['ids']):
  print(f"id : {id} | meta : {meta} | doc : {doc}")
  print(f"meta : {meta}")
  print(f"doc : {doc}")
  print("~~" * 30)
print()

print("저장된 문서 id목록 : ", collection.get(['ids']))
print()

results_vec = collection.get(include=['embeddings'])
# print(results_vec)
first_embedding = results_vec['embeddings'][0]
print(f"임베딩 벡터 차원 수 : {len(first_embedding)}")
print("first_embedding 5개 :",first_embedding[:5])
print()

for id, emb in zip(results_vec['ids'], results_vec['embeddings']):
  print(f"id : {id} | emb(앞 5개) : {emb[:5]}")

{'ids': ['doc1', 'doc2'], 'embeddings': None, 'documents': ['Hello world', 'Chroma is cool'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'source': 'greeting'}, {'source': 'statement'}]}

id : doc1 | meta : {'source': 'greeting'} | doc : Hello world
meta : {'source': 'greeting'}
doc : Hello world
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
id : doc2 | meta : {'source': 'statement'} | doc : Chroma is cool
meta : {'source': 'statement'}
doc : Chroma is cool
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

저장된 문서 id목록 :  {'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}

임베딩 벡터 차원 수 : 384
first_embedding 5개 : [-0.03447729  0.03102317  0.00673498  0.02610895 -0.03936207]

id : doc1 | emb(앞 5개) : [-0.03447729  0.03102317  0.00673498  0.02610895 -0.03936207]
id : doc2 | emb(앞 5개) : [-0.11315749 -0.00511692 -0.01389769 -0.02509309 -0.04774468

In [60]:
# 벡터 기반 검색
# 검색용 질문
query_text = "What's the status of chroma?"
# print(embedding_fn([query_text]))

# 검생 문장도 임베딩을 진행해야 함 - 벡터화
query_embedding = embedding_fn([query_text])[0] # 반드시 list 타입으로 질문해야함!

# Chroma에서 유사 문서 검색 - RAG에서 이 방법을 선행함
search_result = collection.query(
    query_embeddings=[query_embedding], # 질문을 벡터화 시킴
    n_results=2, # 가장 가까운거 두개만
    # 가지고 나올것, distances:유사도 거리(짧을 수록 관련이 깊음 - 0에 근사할 수록 관련 높다)
    include=["documents","metadatas","distances"]
)

for i , (doc, meta, dist) in enumerate(zip(
    search_result['documents'][0],
    search_result['metadatas'][0],
    search_result['distances'][0]
)):
  print(f"결과 {i + 1}번째")
  print(f"  documents : {doc}")
  print(f"  metadatas : {meta}")
  print(f"  distances : {dist:.4f}")
  print("~~" * 20)



결과 1번째
  documents : Chroma is cool
  metadatas : {'source': 'statement'}
  distances : 0.5813
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
결과 2번째
  documents : Hello world
  metadatas : {'source': 'greeting'}
  distances : 1.9883
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
